In [ ]:
!pip install rasterio

In [2]:
import rasterio
import numpy as np

Loading the bands

It loads the raw satellite bands b02,b03,b04,b08 which are used to create dataset for trainig the model.

In [3]:
import rasterio

b02_path = "/content/B02.tiff"
b03_path = "/content/B03.tiff"
b04_path = "/content/B04.tiff"
b08_path = "/content/B08.tiff"

with rasterio.open(b02_path) as src:
    print("B02:", src.count, src.crs)

with rasterio.open(b03_path) as src:
    print("B03:", src.count, src.crs)

with rasterio.open(b04_path) as src:
    print("B04:", src.count, src.crs)

with rasterio.open(b08_path) as src:
    print("B08:", src.count, src.crs)

B02: 1 EPSG:4326
B03: 1 EPSG:4326
B04: 1 EPSG:4326
B08: 1 EPSG:4326


In [4]:
with rasterio.open(b02_path) as src:
    B02 = src.read(1)

with rasterio.open(b03_path) as src:
    B03 = src.read(1)

with rasterio.open(b04_path) as src:
    B04 = src.read(1)

with rasterio.open(b08_path) as src:
    B08 = src.read(1)

print("B03 shape:", B03.shape)
print("B04 shape:", B04.shape)
print("B08 shape:", B08.shape)

B03 shape: (1188, 2500)
B04 shape: (1188, 2500)
B08 shape: (1188, 2500)


Computing NDWI and NDTI : Using the raw satellite bands NDWI and NDTI is calculated
- NDWI helps in identifying water body, Green and NIR reflect differently over water, which helps NDWI  identify water.
- NDTI is special indicator that can give info on turbidity of water, Red and Green reflectance differences help NDTI indicate turbidity.
- Higher NDWI generally indicates greater water presence.


In [39]:
ndwi = np.full(B03.shape, np.nan, dtype="float32")
ndti = np.full(B04.shape, np.nan, dtype="float32")

valid_1 = (B03 + B04) != 0
valid = (B03 + B08) != 0

ndwi[valid] = (B03[valid] - B08[valid]) / (B03[valid] + B08[valid])

ndti[valid_1] = (B04[valid_1] - B03[valid_1]) / (B04[valid_1] + B03[valid_1])

# Replace invalid values with zero
ndwi = np.nan_to_num(ndwi, nan=0.0, posinf=0.0, neginf=0.0)
ndti = np.nan_to_num(ndti, nan=0.0, posinf=0.0, neginf=0.0)

print("NDWI NaN:", np.isnan(ndwi).sum())
print("NDTI NaN:", np.isnan(ndti).sum())

NDWI NaN: 0
NDTI NaN: 0


In [40]:
stacked_data = np.stack([B02, B03, B04, B08, ndwi, ndti], axis=0)

print("Stacked shape:", stacked_data.shape)

Stacked shape: (6, 1188, 2500)


Cutting 32×32 patches

- we cut the image as the model learns local patterns and uses less memory.
- Each patch has a shape of 6,32,32 — 6 channels, each with 32×32 pixels.
- Each patch gets its NTU label by mapping centre coordinates of each patch, to the nearest sensor and uses that sensor's NTU value.
- Neighboring patches may use same NTU value as they can use the same sensor's NTU value and instead of randomly splitting patches, keep a separate image region (such as the rightmost 20%) for validation.

In [41]:
patch = stacked_data[:, :32, :32]

print('Patch shape:', patch.shape)

Patch shape: (6, 32, 32)


In [42]:
patches = []
coords = []


for row in range(0, 1188 - 32 + 1, 32):
    for col in range(0, 2500 - 32 + 1, 32):

        patch = stacked_data[:, row:row+32, col:col+32]

        patches.append(patch)
        coords.append((row, col))

print("Total patches:", len(patches))

print("Patch 1 shape:", patches[0].shape)
print("Patch 2 shape:", patches[1].shape)
print("Patch 3 shape:", patches[2].shape)

Total patches: 2886
Patch 1 shape: (6, 32, 32)
Patch 2 shape: (6, 32, 32)
Patch 3 shape: (6, 32, 32)


In [43]:
import torch
from torch.utils.data import Dataset

In [44]:
from rasterio.transform import xy

def pixel_to_latlon(transform, row, col):
    lon, lat = xy(transform, row, col, offset="center")
    return lat, lon

Dataset class and NTU labels

- __init__: Stores references to patches, coordinates, transform and sensors, __len__: Returns the total number of samples., __getitem__ : For a given index, retrieves the patch, calculates its centre latitude and longitude, finds the nearest sensor using Haversine, and returns the patch and sensor's NTU value as tensors.
- Haversine : Calculates great-circle distance in kilometres between two geographical coordinates.
- Patches and NTU labels are converted to tensors as pytorch model uses tensor variables.
- Tensors allow PyTorch to use autograd for gradient calculations and support GPU computation.

In [45]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    dlat = lat2 - lat1
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


def nearest_sensor(lat, lon, sensors):
    best_sensor = None
    best_distance = float("inf")

    for sensor in sensors:
        distance = haversine_km(
            lat,
            lon,
            sensor["lat"],
            sensor["lon"]
        )

        if distance < best_distance:
            best_distance = distance
            best_sensor = sensor

    return best_sensor

In [46]:
class FusionDataset(Dataset):

    def __init__(self, patches, coords, transform, sensors):
        self.patches = patches
        self.coords = coords
        self.transform = transform
        self.sensors = sensors

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):

        patch = self.patches[index]

        row, col = self.coords[index]

        center_row = row + 16
        center_col = col + 16

        lat, lon = pixel_to_latlon(
            self.transform,
            center_row,
            center_col
        )

        sensor = nearest_sensor(
            lat,
            lon,
            self.sensors
        )

        ntu = sensor["ntu"]

        patch = torch.tensor(
            patch,
            dtype=torch.float32
        )

        ntu = torch.tensor(
            ntu,
            dtype=torch.float32
        )

        return patch, ntu

In [47]:
import rasterio

with rasterio.open("/content/Jul_polluted.tif") as src:
    transform = src.transform

In [48]:
import csv

sensors = []

with open("/content/Jul_sensors.csv", newline="", encoding="utf-8") as f:
    for rec in csv.DictReader(f):
        sensors.append({
            "station_id": rec["Station_ID"],
            "lat": float(rec["Lat"]),
            "lon": float(rec["Lon"]),
            "ntu": float(rec["Turbidity_NTU"])
        })

Dataset is created

- The fusion dataset which is created contains four main parameters: patches - contains image patches, coords - contains pixel positions, transform converts pixel positions into geographical coordinates, and sensors contains sensor locations and NTU readings.
- Total length of dataset is 2886 samples
- Dataset[0] retrieves one patch tensor of shape (6, 32, 32) and its NTU value. The label is found inside __getitem__ when you request an index, not when the dataset is created.

In [49]:
dataset = FusionDataset(
    patches,
    coords,
    transform,
    sensors
)

print("Number of samples:", len(dataset))

patch, ntu = dataset[0]

print("Patch type:", type(patch))
print("Patch shape:", patch.shape)

print("NTU type:", type(ntu))
print("NTU value:", ntu)

Number of samples: 2886
Patch type: <class 'torch.Tensor'>
Patch shape: torch.Size([6, 32, 32])
NTU type: <class 'torch.Tensor'>
NTU value: tensor(5.)


CNN architecture

- Conv2d uses a small window over the satellite image to learn features and converts 6 input channels to 32 output channels.
- MaxPool2d compares groups of data and takes only the highest vaue and ReLU converts negative vaue to zero and leaves positive number as it is to help CNN learn complex patterns.
- Flatten cahnges the sahpe by converting it into a one long vector.
- Linear takes 64 × 8 × 8 inputs and produces one NTU prediction i.e 4096 by multiplying number of feature maps and height, width of each map.
- (6, 32, 32) → conv1 → (32, 32, 32) → pool → (32, 16, 16) → conv2 → (64, 16, 16) → pool → (64, 8, 8) → flatten → 4096 → Linear → 1

In [50]:
import torch
import torch.nn as nn


class NTUCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=6,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.fc = nn.Linear(64 * 8 * 8, 1)

    def forward(self, x):

        x = torch.relu(self.conv1(x))
        x = self.pool(x)

        x = torch.relu(self.conv2(x))
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        return x

In [51]:
import torch

model = NTUCNN()
x = torch.randn(1, 6, 32, 32)

print("Input:", x.shape)

x = torch.relu(model.conv1(x))
print("After Conv1 + ReLU:", x.shape)

x = model.pool(x)
print("After MaxPool1:", x.shape)

x = torch.relu(model.conv2(x))
print("After Conv2 + ReLU:", x.shape)

x = model.pool(x)
print("After MaxPool2:", x.shape)

x = torch.flatten(x, start_dim=1)
print("After Flatten:", x.shape)

x = model.fc(x)
print("After Linear:", x.shape)

Input: torch.Size([1, 6, 32, 32])
After Conv1 + ReLU: torch.Size([1, 32, 32, 32])
After MaxPool1: torch.Size([1, 32, 16, 16])
After Conv2 + ReLU: torch.Size([1, 64, 16, 16])
After MaxPool2: torch.Size([1, 64, 8, 8])
After Flatten: torch.Size([1, 4096])
After Linear: torch.Size([1, 1])


In [52]:
patch, ntu = dataset[0]

output = model(patch.unsqueeze(0))

print("Input shape:", patch.shape)
print("Output shape:", output.shape)
print("Predicted NTU:", output)

Input shape: torch.Size([6, 32, 32])
Output shape: torch.Size([1, 1])
Predicted NTU: tensor([[-0.0319]], grad_fn=<AddmmBackward0>)


In [53]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

In [54]:
images, labels = next(iter(loader))
labels = labels.unsqueeze(1)

outputs = model(images)

print("Input shape:", images.shape)
print("Label shape:", labels.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([8, 6, 32, 32])
Label shape: torch.Size([8, 1])
Output shape: torch.Size([8, 1])


Loss function, optimizer and label standardization

- Mean Squared Error (MSE) calculates how far the model's predictions are from the actual NTU values.
- Adam optimizer is used to update the parameters it adapts the learning rate for each parameter based on estimates of past gradients, this makes training more efficient.
- We use only training-set mean and standard deviation to standardize labels because if we use validation data it may be leaked to the model and affects the final validation.
- As after standardization, the training labels have mean approximately 0 and standard deviation approximately 1 , an MSE near 1.0 means the model is doing about as well as predicting the mean

In [55]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [56]:
loss_before = criterion(outputs, labels)

print("Loss before:", loss_before.item())

optimizer.zero_grad()

loss_before.backward()

optimizer.step()

with torch.no_grad():
    outputs_after = model(images)
    loss_after = criterion(outputs_after, labels)

print("Loss after:", loss_after.item())

Loss before: 37184.64453125
Loss after: 37116.625


In [57]:
images, labels = next(iter(loader))

with torch.no_grad():
    outputs = model(images)

print("========== INPUT STATISTICS ==========")

for i in range(images.shape[1]):
    channel = images[:, i, :, :]

    print(f"\nChannel {i+1}:")
    print("Min:", channel.min().item())
    print("Max:", channel.max().item())
    print("Mean:", channel.mean().item())

print("\n========== LABEL STATISTICS ==========")
print("Min:", labels.min().item())
print("Max:", labels.max().item())
print("Mean:", labels.mean().item())

print("\n========== MODEL OUTPUT STATISTICS ==========")
print("Min:", outputs.min().item())
print("Max:", outputs.max().item())
print("Mean:", outputs.mean().item())

========== INPUT STATISTICS ==========

Channel 1:
Min: 0.0
Max: 1.3487999439239502
Mean: 0.15533606708049774

Channel 2:
Min: 0.010499999858438969
Max: 1.1288000345230103
Mean: 0.17539219558238983

Channel 3:
Min: 0.017999999225139618
Max: 0.9872000217437744
Mean: 0.16311965882778168

Channel 4:
Min: 0.028999999165534973
Max: 0.848800003528595
Mean: 0.32186150550842285

Channel 5:
Min: -0.8628347516059875
Max: 0.25790753960609436
Mean: -0.3588658571243286

Channel 6:
Min: -0.6161187291145325
Max: 0.45583376288414
Mean: -0.04778038337826729

========== LABEL STATISTICS ==========
Min: 5.0
Max: 293.6000061035156
Mean: 199.10501098632812

========== MODEL OUTPUT STATISTICS ==========
Min: 0.16662107408046722
Max: 0.22010047733783722
Mean: 0.19365692138671875


In [58]:
from torch.utils.data import random_split

# 1. Calculate split sizes
total_size = len(dataset)

train_size = int(0.8 * total_size)
val_size = total_size - train_size

# 2. Split dataset
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# 3. Print split sizes
print("Total samples:", total_size)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

# 4. Get NTU labels from training dataset only
train_labels = torch.tensor([
    train_dataset[i][1].item()
    for i in range(len(train_dataset))
])

# 5. Calculate training label statistics
label_mean = train_labels.mean()
label_std = train_labels.std()

print("\nTraining label mean:", label_mean.item())
print("Training label std:", label_std.item())

Total samples: 2886
Training samples: 2308
Validation samples: 578

Training label mean: 180.21112060546875
Training label std: 97.36724853515625


In [60]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

In [61]:
model = NTUCNN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [62]:
for epoch in range(5):

    model.train()
    train_loss = 0.0

    for images, labels in train_loader:

        labels = labels.unsqueeze(1)
        labels = (labels - label_mean) / label_std

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    print(f"Epoch {epoch+1}/5")
    print("Training Loss:", train_loss)

Epoch 1/5
Training Loss: 0.8229441504319647
Epoch 2/5
Training Loss: 0.6619171141340658
Epoch 3/5
Training Loss: 0.6432038123065212
Epoch 4/5
Training Loss: 0.5975851081698411
Epoch 5/5
Training Loss: 0.5623790904570203


In [63]:
model.eval()
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        labels = labels.unsqueeze(1)
        labels = (labels - label_mean) / label_std

        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()

val_loss = val_loss / len(val_loader)
print("Validation Loss:", val_loss)

Validation Loss: 0.6457964643223645


The model was trained for 5 epochs using standardized NTU labels. Training loss decreased over the epochs, and validation loss reached approximately 0.646. However, random splitting may cause spatial data leakage because neighboring patches can appear in both sets. Therefore, spatial validation is needed for a more reliable evaluation.